Evaluating the disagreement between median mosaicks and harmonic mosaics to get per band RMSE at random pixels. this is just for 2018 - needs to be done for every year to generate table of per year RMSE between the two approaches. 

In [1]:
from pathlib import Path
import numpy as np
import rasterio
from rasterio.windows import Window

# =============================================================================
# SETTINGS
# =============================================================================

YEAR = 2018

MONTHLY_DIR = Path(r"E:\NCA_DATA_backup_20260409\S2_Harmonics\S2_SR_monthly_median_mosaicked")
HARM_PATH   = Path(r"C:\NCA_DATA\S2_Harmonics_vrt\S2SR_Harmonics_2018_4th_amp_phase_masked.tif")

# Monthly mosaics contain these exported bands in fixed order
MONTHLY_BAND_MAP = {
    "B2": 1,
    "B3": 2,
    "B4": 3,
    "B5": 4,
    "B6": 5,
    "B7": 6,
    "B8": 7,
    "B8A": 8,
    "B11": 9,
    "B12": 10,
    "nObs": 11,
}

# Validate all reflectance bands that exist in the monthly mosaics
TARGET_VARS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]

WINDOW_WIDTH = 100
WINDOW_HEIGHT = 100

DAYS_IN_YEAR = 365.0
MONTH_MID_DOY = np.array([15, 45, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349], dtype=np.float32)
T_MONTH = MONTH_MID_DOY / DAYS_IN_YEAR

# =============================================================================
# HELPERS
# =============================================================================

def list_band_descriptions(ds):
    out = []
    for i in range(1, ds.count + 1):
        d = ds.descriptions[i - 1]
        if d is None or str(d).strip() == "":
            d = f"band_{i}"
        out.append(str(d).strip())
    return out

def normalize_name(s):
    return (
        str(s).strip()
        .replace(" ", "")
        .replace("-", "")
        .replace(".", "")
        .replace("/", "")
        .replace("_", "")
        .lower()
    )

def find_harmonic_band_indices(harm_desc, target_var):
    target = normalize_name(target_var)

    needed = {
        "beta0": [f"beta0{target}", f"constant{target}", f"{target}beta0", f"{target}constant"],
        "A1":    [f"a1{target}", f"{target}a1"],
        "phi1":  [f"phi1{target}", f"{target}phi1"],
        "A2":    [f"a2{target}", f"{target}a2"],
        "phi2":  [f"phi2{target}", f"{target}phi2"],
        "A3":    [f"a3{target}", f"{target}a3"],
        "phi3":  [f"phi3{target}", f"{target}phi3"],
        "A4":    [f"a4{target}", f"{target}a4"],
        "phi4":  [f"phi4{target}", f"{target}phi4"],
    }

    out = {}

    for key, candidates in needed.items():
        found = None

        # exact / contains lookup
        for i, d in enumerate(harm_desc, start=1):
            nd = normalize_name(d)
            if any(c == nd or c in nd for c in candidates):
                found = i
                break

        if found is None:
            raise RuntimeError(f"Could not find harmonic band '{key}' for variable '{target_var}'")

        out[key] = found

    return out

def read_window_band(path, band_index, window):
    with rasterio.open(path) as ds:
        arr = ds.read(band_index, window=window).astype(np.float32)
        nodata = ds.nodata
    if nodata is not None:
        arr[arr == nodata] = np.nan
    return arr

def harmonic_reconstruct(beta0, A, phi, t):
    """
    y(t) = beta0 + sum_{k=1}^4 A_k * cos(2*pi*k*t - phi_k)

    beta0: (N,)
    A:     (N, 4)
    phi:   (N, 4)
    t:     (12,)
    """
    y = np.repeat(beta0[:, None], len(t), axis=1).astype(np.float32)
    for k in range(4):
        y += A[:, [k]] * np.cos(2.0 * np.pi * (k + 1) * t[None, :] - phi[:, [k]])
    return y

# =============================================================================
# FILES AND WINDOW
# =============================================================================

monthly_files = sorted(MONTHLY_DIR.glob(f"S2_SR_{YEAR}_*_median_mosaic.tif"))
if len(monthly_files) != 12:
    raise RuntimeError(f"Expected 12 monthly files for {YEAR}, found {len(monthly_files)}")

with rasterio.open(monthly_files[0]) as ds_m:
    month_desc = list_band_descriptions(ds_m)
    height, width = ds_m.height, ds_m.width
    month_transform = ds_m.transform
    month_crs = ds_m.crs
    month_count = ds_m.count

with rasterio.open(HARM_PATH) as ds_h:
    harm_desc = list_band_descriptions(ds_h)
    harm_height, harm_width = ds_h.height, ds_h.width
    harm_transform = ds_h.transform
    harm_crs = ds_h.crs

if (harm_height, harm_width) != (height, width):
    raise RuntimeError("Shape mismatch between monthly mosaic and harmonic raster")
if harm_transform != month_transform:
    raise RuntimeError("Transform mismatch between monthly mosaic and harmonic raster")
if harm_crs != month_crs:
    raise RuntimeError("CRS mismatch between monthly mosaic and harmonic raster")

if month_count != 11:
    print(f"Warning: expected 11 monthly bands, found {month_count}")

col_off = (width - WINDOW_WIDTH) // 2
row_off = (height - WINDOW_HEIGHT) // 2

if col_off < 0 or row_off < 0:
    raise RuntimeError("Requested center window is larger than the raster dimensions")

window = Window(col_off=col_off, row_off=row_off, width=WINDOW_WIDTH, height=WINDOW_HEIGHT)

print(f"Center window: row_off={row_off}, col_off={col_off}, width={WINDOW_WIDTH}, height={WINDOW_HEIGHT}")
print(f"Monthly files found: {len(monthly_files)}")

# =============================================================================
# VALIDATION
# =============================================================================

for var in TARGET_VARS:
    print("\n" + "=" * 70)
    print(var)
    print("=" * 70)

    monthly_band = MONTHLY_BAND_MAP[var]
    harm_idx = find_harmonic_band_indices(harm_desc, var)

    # -------------------------------------------------------------------------
    # Read monthly values for this band
    # -------------------------------------------------------------------------
    monthly_cube = np.stack(
        [read_window_band(f, monthly_band, window) for f in monthly_files],
        axis=0
    )  # (12, H, W)

    # -------------------------------------------------------------------------
    # Read harmonic coefficients
    # -------------------------------------------------------------------------
    beta0 = read_window_band(HARM_PATH, harm_idx["beta0"], window).reshape(-1)

    A = np.stack([
        read_window_band(HARM_PATH, harm_idx["A1"], window).reshape(-1),
        read_window_band(HARM_PATH, harm_idx["A2"], window).reshape(-1),
        read_window_band(HARM_PATH, harm_idx["A3"], window).reshape(-1),
        read_window_band(HARM_PATH, harm_idx["A4"], window).reshape(-1),
    ], axis=1)

    phi = np.stack([
        read_window_band(HARM_PATH, harm_idx["phi1"], window).reshape(-1),
        read_window_band(HARM_PATH, harm_idx["phi2"], window).reshape(-1),
        read_window_band(HARM_PATH, harm_idx["phi3"], window).reshape(-1),
        read_window_band(HARM_PATH, harm_idx["phi4"], window).reshape(-1),
    ], axis=1)

    # -------------------------------------------------------------------------
    # Reconstruct and compute residuals
    # -------------------------------------------------------------------------
    monthly_vals = monthly_cube.reshape(12, -1).T   # (N, 12)
    harmonic_vals = harmonic_reconstruct(beta0, A, phi, T_MONTH)

    valid = np.all(np.isfinite(monthly_vals), axis=1) & np.all(np.isfinite(harmonic_vals), axis=1)
    monthly_vals = monthly_vals[valid]
    harmonic_vals = harmonic_vals[valid]

    if monthly_vals.shape[0] == 0:
        print("No valid pixels in window.")
        continue

    residuals = monthly_vals - harmonic_vals   # observed - fitted
    flat_res = residuals.reshape(-1)

    mean_res = np.nanmean(flat_res)
    med_res = np.nanmedian(flat_res)
    rmse = np.sqrt(np.nanmean(flat_res ** 2))
    mae = np.nanmean(np.abs(flat_res))
    q05, q25, q50, q75, q95 = np.nanpercentile(flat_res, [5, 25, 50, 75, 95])

    monthly_mean_res = np.nanmean(residuals, axis=0)
    monthly_rmse = np.sqrt(np.nanmean(residuals ** 2, axis=0))

    print(f"Valid pixels:        {monthly_vals.shape[0]}")
    print(f"Mean residual:      {mean_res:.6f}")
    print(f"Median residual:    {med_res:.6f}")
    print(f"RMSE:               {rmse:.6f}")
    print(f"Mean abs residual:  {mae:.6f}")
    print(f"Residual quantiles: 5%={q05:.6f}, 25%={q25:.6f}, 50%={q50:.6f} 75%={q75:.6f}, 95%={q95:.6f}")

    print("\nMonthly residual summary:")
    for m in range(12):
        print(
            f"  Month {m+1:02d}  "
            f"mean_res={monthly_mean_res[m]: .6f}  "
            f"rmse={monthly_rmse[m]:.6f}"
        )
        # -------------------------------------------------------------------------
# NORMALIZED RMSE (per pixel, then summarized)
# -------------------------------------------------------------------------

# seasonal amplitude per pixel (observed)
amp = np.nanmax(monthly_vals, axis=1) - np.nanmin(monthly_vals, axis=1)

# avoid divide-by-zero
amp_safe = np.where(amp == 0, np.nan, amp)

# per-pixel RMSE
pixel_rmse = np.sqrt(np.nanmean((monthly_vals - harmonic_vals) ** 2, axis=1))

# normalized RMSE per pixel
nrmse = pixel_rmse / amp_safe

# summarize
mean_nrmse = np.nanmean(nrmse)
median_nrmse = np.nanmedian(nrmse)
q05, q25, q75, q95 = np.nanpercentile(nrmse, [5, 25, 75, 95])

print("\nNormalized RMSE (relative to seasonal amplitude):")
print(f"Mean nRMSE:       {mean_nrmse:.4f}")
print(f"Median nRMSE:     {median_nrmse:.4f}")
print(f"Quantiles:        5%={q05:.4f}, 25%={q25:.4f}, 75%={q75:.4f}, 95%={q95:.4f}")

        

RuntimeError: Expected 12 monthly files for 2018, found 0